# P09B — Augmentasi, ensemble, Green ML, dan AutoML

Jalankan P09A dahulu. Dataset spalling berlabel sintetis. Pembagian train/test dilakukan sebelum augmentasi; data uji tidak diubah.

In [ ]:
from pathlib import Path
import numpy as np,pandas as pd,time
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier,HistGradientBoostingClassifier,VotingClassifier
from sklearn.metrics import precision_score,recall_score,f1_score,balanced_accuracy_score,roc_auc_score
from sklearn.base import clone
p=Path('Bab9_Paket/Data/rc_spalling_synthetic.csv'); df=pd.read_csv(p); print(df.shape,df.spalling.value_counts(normalize=True).round(3))
features=['width_mm','steel_ratio_pct','fc_mpa','cover_mm','load_ratio','fire_minutes']; X=df[features].to_numpy(); y=(df.spalling=='Spalling').astype(int).to_numpy()
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=42,stratify=y)
print('minority train/test:',int(ytr.sum()),int(yte.sum()),'| test prevalence preserved:',round(yte.mean(),3))
# SMOTE-like linear interpolation used only on training fold. This is a teaching approximation, not an imbalanced-learn implementation.
def interpolate_minority(X,y,seed=42):
 r=np.random.default_rng(seed); ix=np.flatnonzero(y==1); needed=int((y==0).sum()-len(ix)); rows=[]
 for _ in range(needed):
  a,b=r.choice(ix,2,replace=True); lam=r.random(); rows.append(X[a]+lam*(X[b]-X[a]))
 return np.vstack([X,np.asarray(rows)]),np.r_[y,np.ones(len(rows),dtype=int)]
Xbal,ybal=interpolate_minority(Xtr,ytr); print('training counts before/after:',np.bincount(ytr),np.bincount(ybal),'test unchanged:',len(yte))
models={'Logistic balanced':make_pipeline(StandardScaler(),LogisticRegression(class_weight='balanced',max_iter=1500,random_state=42)), 'Logistic + interpolation':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1500,random_state=42)), 'Random Forest':RandomForestClassifier(n_estimators=160,min_samples_leaf=3,class_weight='balanced',random_state=42,n_jobs=1), 'Gradient Boosting':HistGradientBoostingClassifier(max_iter=120,l2_regularization=1.0,random_state=42)}
results=[]; times={}
for name,model in models.items():
 start=time.perf_counter()
 if name.endswith('interpolation'): model.fit(Xbal,ybal)
 else: model.fit(Xtr,ytr)
 times[name]=time.perf_counter()-start; pr=model.predict(Xte); prob=model.predict_proba(Xte)[:,1]
 results.append({'model':name,'precision':precision_score(yte,pr,zero_division=0),'recall':recall_score(yte,pr,zero_division=0),'f1':f1_score(yte,pr,zero_division=0),'balanced_accuracy':balanced_accuracy_score(yte,pr),'roc_auc':roc_auc_score(yte,prob),'fit_seconds':times[name]})
print(pd.DataFrame(results).round(3).to_string(index=False))
# Soft-voting ensemble: all base learners trained on original training observations.
vote=VotingClassifier(estimators=[('lr',clone(models['Logistic balanced'])),('rf',clone(models['Random Forest'])),('gb',clone(models['Gradient Boosting']))],voting='soft')
t=time.perf_counter(); vote.fit(Xtr,ytr); tv=time.perf_counter()-t; vp=vote.predict(Xte); vprob=vote.predict_proba(Xte)[:,1]
print('soft voting:',{'precision':round(precision_score(yte,vp,zero_division=0),3),'recall':round(recall_score(yte,vp,zero_division=0),3),'f1':round(f1_score(yte,vp,zero_division=0),3),'roc_auc':round(roc_auc_score(yte,vprob),3),'fit_seconds':round(tv,4)})
# Search model hyperparameters on training fold only; test fold stays held out.
search=GridSearchCV(RandomForestClassifier(class_weight='balanced',random_state=42,n_jobs=1),{'n_estimators':[80,160],'max_depth':[None,5,9],'min_samples_leaf':[2,5]},scoring='f1',cv=3,n_jobs=1,refit=True)
t=time.perf_counter(); search.fit(Xtr,ytr); search_s=time.perf_counter()-t
best=search.best_estimator_; bp=best.predict(Xte)
print('GridSearchCV train-CV best:',search.best_params_,'CV F1:',round(search.best_score_,3),'test F1 once:',round(f1_score(yte,bp,zero_division=0),3),'search seconds:',round(search_s,3))
# Whole-pipeline energy proxy includes baseline/ensemble fits and the complete grid search; not metered.
pipeline_seconds=sum(times.values())+tv+search_s
for watts in [25,60,120]:
 kwh=watts*pipeline_seconds/3_600_000
 print(f'whole-pipeline proxy at {watts} W: {kwh:.8g} kWh; at 0.42 kgCO2e/kWh: {kwh*.42:.8g} kgCO2e')